In [ ]:
# 1. Setup: datasets, folders and training settings
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from ultralytics import YOLO

# main dataset: train, tune and test on it
WORKSPACE, PROJECT, VERSION = "fruit-onxba", "rotten-fresh-fruitclassification", 2   # v2 = the 1,519 original images
# second dataset, photos from other people: never trained on, only used to check the model on unseen photos
EXT_WORKSPACE, EXT_PROJECT, EXT_VERSION = "objectdetectionlearning-4empo", "food-freshness-classification-zu3j5", 1
DATA_DIR = Path("datasets/fruit").resolve()        # downloads are git-ignored
EXT_DIR = Path("datasets/external").resolve()
WEIGHTS = Path("weights/fruit_yolo11n_cls.pt")     # committed, loaded by the app
OUT = Path("artifacts")                            # results the app reads
OUT.mkdir(exist_ok=True)

SEED = 42
EPOCHS, IMGSZ, BATCH = 20, 224, 32
SPLITS = ["train", "valid", "test"]
IMAGE_TYPES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
NONE = "no fruit found"

In [ ]:
# 2. Download both datasets from Roboflow Universe (skipped if already on disk)
def download(workspace, project, version, folder):
    if not folder.exists():
        from roboflow import Roboflow

        from api_key import load_api_key

        rf = Roboflow(api_key=load_api_key())
        rf.workspace(workspace).project(project).version(version).download("folder", location=str(folder))


download(WORKSPACE, PROJECT, VERSION, DATA_DIR)
download(EXT_WORKSPACE, EXT_PROJECT, EXT_VERSION, EXT_DIR)

# a class that is not in every split cannot be learned or tested ("Unlabeled": 1 train image) -> remove it
for split in SPLITS:
    for folder in (DATA_DIR / split).iterdir():
        if folder.is_dir() and not all((DATA_DIR / s / folder.name).is_dir() for s in SPLITS):
            print(f"removing {split}/{folder.name} ({len(list(folder.iterdir()))} image)")
            shutil.rmtree(folder)


def list_images(root):
    return pd.DataFrame([
        {"split": split.name, "label": cls.name, "path": str(p)}
        for split in sorted(root.iterdir()) if split.is_dir()
        for cls in sorted(split.iterdir()) if cls.is_dir()
        for p in sorted(cls.iterdir()) if p.suffix.lower() in IMAGE_TYPES])


images = list_images(DATA_DIR)
CLASSES = sorted(images.label.unique())
ROTTEN = [c for c in CLASSES if "rotten" in c.lower()]          # positive class = any rotten fruit
FRESH = [c for c in CLASSES if c not in ROTTEN]


def fruit(name):
    return name.lower().replace("fresh", "").replace("rotten", "").strip(" _-").rstrip("s")


# the second dataset names its classes differently ("Rotten Apple"): map them onto ours
KEY = {("rotten" in c.lower(), fruit(c)): c for c in CLASSES}
external = list_images(EXT_DIR)
external["label"] = [KEY.get(("rotten" in c.lower(), fruit(c))) for c in external.label]
external = external.dropna(subset=["label"]).assign(split="external")
print(CLASSES, "| external photos:", len(external))

In [ ]:
# 3. Describe the dataset: images per class and split, image size, near-duplicates
counts = images.pivot_table(index="label", columns="split", values="path", aggfunc="count", fill_value=0)[SPLITS]
counts["external"] = external.label.value_counts()
counts.loc["total"] = counts.sum()
print(counts)

images["width"], images["height"] = zip(*[Image.open(p).size for p in images.path])
print(images.groupby("split")[["width", "height"]].median())


def average_hash(path):
    # 8x8 greyscale thumbnail -> 64 bits; identical hashes = (near) identical pictures
    small = np.asarray(Image.open(path).convert("L").resize((8, 8)), dtype=float)
    return "".join("1" if v else "0" for v in (small > small.mean()).ravel())


images["hash"] = [average_hash(p) for p in images.path]
external["hash"] = [average_hash(p) for p in external.path]
train_hashes = set(images.loc[images.split == "train", "hash"])
duplicates = {s: int(images[images.split == s].hash.isin(train_hashes).sum()) for s in ["valid", "test"]}
duplicates["external"] = int(external.hash.isin(train_hashes).sum())
print("images that also appear in train:", duplicates)

In [ ]:
# 4. Fine-tune yolo11n-cls from the pretrained checkpoint
model = YOLO("yolo11n-cls.pt")
model.train(data=str(DATA_DIR), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED, deterministic=True,
            project=str(Path("runs").resolve()), name="fruit", exist_ok=True, workers=2, plots=False)
WEIGHTS.parent.mkdir(exist_ok=True)
shutil.copy(model.trainer.best, WEIGHTS)
model = YOLO(str(WEIGHTS))
assert list(model.names.values()) == CLASSES

In [ ]:
# 5. Score validation, test and external photos: probability of each class and of "rotten"
def score(part):
    rows = []
    results = model.predict(list(part.path), imgsz=IMGSZ, verbose=False, stream=True)
    for (_, img), r in zip(part.iterrows(), results):
        probs = r.probs.data.cpu().numpy()
        rows.append({"split": img.split, "path": img.path, "label": img.label,
                     "pred": CLASSES[int(probs.argmax())], **{f"p_{c}": float(q) for c, q in zip(CLASSES, probs)}})
    d = pd.DataFrame(rows)
    d["is_rotten"] = d.label.isin(ROTTEN).astype(int)
    d["p_rotten"] = d[[f"p_{c}" for c in ROTTEN]].sum(axis=1)
    return d


scores = pd.concat([score(images[images.split.isin(["valid", "test"])]), score(external)], ignore_index=True)
val, test, ext = (scores[scores.split == s].reset_index(drop=True) for s in ["valid", "test", "external"])
print(f"accuracy: validation {(val.pred == val.label).mean():.3f}, test {(test.pred == test.label).mean():.3f}, "
      f"external photos {(ext.pred == ext.label).mean():.3f}")

In [ ]:
# 6. Baselines on the same test images: majority class, and the COCO model zero-shot
majority = images[images.split == "train"].label.value_counts().idxmax()
test["pred_majority"] = majority

# COCO knows apples, bananas and oranges but has no idea of "rotten": the most it can say is
# "this is an apple", which we read as "fresh apple". If it finds no fruit, it has no answer.
coco = YOLO("yolo11n.pt")
COCO_FRUIT = {"apple", "banana", "orange"}
zero_shot, box_share = [], []
for r in coco.predict(list(test.path), conf=0.1, verbose=False, stream=True):
    best = None
    for box, cls, conf in zip(r.boxes.xywh.cpu().numpy(), r.boxes.cls.cpu().numpy(), r.boxes.conf.cpu().numpy()):
        name = coco.names[int(cls)]
        if name in COCO_FRUIT and (best is None or conf > best[2]):
            best = (name, box, conf)
    if best is None:
        zero_shot.append(NONE)
        box_share.append(np.nan)
    else:
        zero_shot.append(KEY[(False, best[0])])
        box_share.append(best[1][2] * best[1][3] / (r.orig_shape[0] * r.orig_shape[1]))
test["pred_zero_shot"] = zero_shot
test["fruit_box_share"] = box_share      # size of the fruit relative to the image
print(f"COCO found a fruit in {test.fruit_box_share.notna().mean():.0%} of test images; "
      f"median fruit size = {test.fruit_box_share.median():.0%} of the image")

In [ ]:
# 7. Per-class precision and recall, confusion matrices, overall accuracy
EVALUATIONS = {"Fine-tuned yolo11n-cls": (test, "pred"),
               "Baseline: majority class": (test, "pred_majority"),
               "Baseline: COCO zero-shot": (test, "pred_zero_shot"),
               "Fine-tuned, external photos": (ext, "pred")}
PRED_LABELS = CLASSES + [NONE]


def per_class(y, pred):
    out = []
    for c in CLASSES:
        tp = ((pred == c) & (y == c)).sum()
        out.append({"class": c, "support": int((y == c).sum()),
                    "precision": tp / (pred == c).sum() if (pred == c).sum() else 0.0,
                    "recall": tp / (y == c).sum()})
    return out


def rotten_vs_fresh(y, pred):
    actual, flagged = y.isin(ROTTEN), pred.isin(ROTTEN)
    tp = (actual & flagged).sum()
    return {"rotten_precision": tp / flagged.sum() if flagged.sum() else 0.0, "rotten_recall": tp / actual.sum()}


per_class_table = pd.DataFrame([{"model": m, **row} for m, (d, col) in EVALUATIONS.items()
                                for row in per_class(d.label, d[col])])
overall = pd.DataFrame([{"model": m, "images": len(d), "accuracy": (d[col] == d.label).mean(),
                         "fruit_type_accuracy": (d[col].map(fruit) == d.label.map(fruit)).mean(),
                         **rotten_vs_fresh(d.label, d[col])} for m, (d, col) in EVALUATIONS.items()])
confusion = {m: pd.crosstab(pd.Categorical(d.label, CLASSES), pd.Categorical(d[col], PRED_LABELS),
                            dropna=False).values.tolist() for m, (d, col) in EVALUATIONS.items()}
print(overall.round(3).to_string(index=False))
print(per_class_table[per_class_table.model.str.startswith("Fine")].round(3).to_string(index=False))

In [ ]:
# 8. Business assumptions (each one is justified in REPORT.md)
COST = {"missed_rotten": 2.00,    # FN: rotten fruit packed -> mould spreads to its neighbours in the box, claims
        "discarded_good": 0.20,   # FP: a good fruit thrown away -> ~0.2 kg at ~€0.95/kg packed price
        "caught_rotten": 0.0,     # TP: rotten fruit removed -> no extra cost (it was worthless anyway)
        "passed_good": 0.0}       # TN: good fruit packed -> normal business
BASE_RATE = 0.03                  # share of fruit on the real line that is rotten (studies: under 3.5%)
FRUIT_PER_DAY = 200_000           # one line, ~5 tonnes/hour for an 8-hour shift, ~0.2 kg per fruit
SECONDS_PER_REVIEW = 3            # pick the flagged fruit up, turn it, decide
WAGE_PER_HOUR = 13.0              # packing worker ~€10/hour + employer social security (~30%)
SHIFT_HOURS = 8
REVIEWERS_AVAILABLE = 1           # one worker on the review station
REVIEW_COST = WAGE_PER_HOUR * SECONDS_PER_REVIEW / 3600
MAX_REVIEW_SHARE = REVIEWERS_AVAILABLE * SHIFT_HOURS * 3600 / SECONDS_PER_REVIEW / FRUIT_PER_DAY
print(f"one review costs €{REVIEW_COST:.4f}; staff can review at most {MAX_REVIEW_SHARE:.1%} of fruit")

In [ ]:
# 9. Cost against threshold: reject a fruit when p_rotten >= threshold (the model decides alone)
THRESHOLDS = np.round(np.arange(0.01, 1.0, 0.01), 2)


def band_cost(d, low, high, base_rate=BASE_RATE, cost=COST):
    # expected € per fruit: pass if p < low, fail if p >= high, a person checks everything in between.
    # Rates come from the images; base_rate re-weights them to the real line (the dataset is ~65% rotten).
    r, f = d.p_rotten[d.is_rotten == 1].to_numpy(), d.p_rotten[d.is_rotten == 0].to_numpy()
    def shares(p):
        return (p < low).mean(), ((p >= low) & (p < high)).mean(), (p >= high).mean()
    r_pass, r_review, r_fail = shares(r)
    f_pass, f_review, f_fail = shares(f)
    review_share = base_rate * r_review + (1 - base_rate) * f_review
    per_fruit = (base_rate * (r_pass * cost["missed_rotten"] + (r_review + r_fail) * cost["caught_rotten"])
                 + (1 - base_rate) * (f_fail * cost["discarded_good"] + (f_pass + f_review) * cost["passed_good"])
                 + review_share * REVIEW_COST)
    return per_fruit, review_share


def best_of(candidates):
    # lowest cost; if several thresholds tie, take the middle one (furthest from both edges)
    costs = np.array([c for c, _ in candidates.values()])
    tied = [k for k, (c, _) in candidates.items() if np.isclose(c, costs.min())]
    return tied[len(tied) // 2]


sweep = pd.DataFrame([{"threshold": t, **{f"cost_per_1000_{name}": 1000 * band_cost(d, t, t)[0]
                                          for name, d in [("validation", val), ("test", test), ("external", ext)]}}
                      for t in THRESHOLDS])
THRESHOLD = float(best_of({t: band_cost(val, t, t) for t in THRESHOLDS}))
print(f"threshold with the lowest cost on validation: {THRESHOLD}")

In [ ]:
# 10. Three outcomes: pass / human review / fail. Pick the band with the lowest cost that staff can handle
bands = {(lo, hi): band_cost(val, lo, hi) for lo in THRESHOLDS for hi in THRESHOLDS if lo <= hi}
bands = {k: v for k, v in bands.items() if v[1] <= MAX_REVIEW_SHARE}
LOW, HIGH = (float(x) for x in best_of(bands))
print(f"pass below {LOW}, human review between, fail from {HIGH}")


def policies(d, base_rate=BASE_RATE, cost=COST):
    n = FRUIT_PER_DAY
    rows = [("Do nothing (pack everything)", *band_cost(d, 1.01, 1.01, base_rate, cost)),
            ("A person checks every fruit", *band_cost(d, -0.01, 1.01, base_rate, cost)),
            (f"Model alone (threshold {THRESHOLD})", *band_cost(d, THRESHOLD, THRESHOLD, base_rate, cost)),
            (f"Model + review ({LOW} to {HIGH})", *band_cost(d, LOW, HIGH, base_rate, cost))]
    return pd.DataFrame([{"policy": name, "cost_per_1000": 1000 * c, "cost_per_day": n * c,
                          "reviews_per_day": n * s, "people_needed": n * s * SECONDS_PER_REVIEW / 3600 / SHIFT_HOURS}
                         for name, c, s in rows])


policy_tables = {"test": policies(test), "external": policies(ext)}
for name, table in policy_tables.items():
    print(f"\n{name}:\n", table.round(2).to_string(index=False))

In [ ]:
# 11. Sensitivity: how the best threshold and the costs move if the assumptions are wrong
sensitivity = []
for base_rate in [0.01, 0.03, 0.05, 0.10]:
    for missed in [0.5, 1.0, 2.0, 4.0]:
        cost = {**COST, "missed_rotten": missed}
        t = best_of({t: band_cost(val, t, t, base_rate, cost) for t in THRESHOLDS})
        sensitivity.append({"base_rate": base_rate, "cost_missed_rotten": missed, "best_threshold": t,
                            "do_nothing_per_1000": 1000 * band_cost(test, 1.01, 1.01, base_rate, cost)[0],
                            "model_alone_test_per_1000": 1000 * band_cost(test, t, t, base_rate, cost)[0],
                            "model_alone_external_per_1000": 1000 * band_cost(ext, t, t, base_rate, cost)[0]})
sensitivity = pd.DataFrame(sensitivity)
print(sensitivity.round(2).to_string(index=False))

In [ ]:
# 12. Save everything the app needs (it never trains)
scores.drop(columns="path").to_csv(OUT / "scores.csv", index=False)
sweep.to_csv(OUT / "cost_sweep.csv", index=False)
per_class_table.to_csv(OUT / "per_class.csv", index=False)
sensitivity.to_csv(OUT / "sensitivity.csv", index=False)
summary = {
    "dataset": {"workspace": WORKSPACE, "project": PROJECT, "version": VERSION,
                "external": {"workspace": EXT_WORKSPACE, "project": EXT_PROJECT, "version": EXT_VERSION},
                "counts": counts.reset_index().to_dict("records"),
                "median_image_size": images.groupby("split")[["width", "height"]].median().to_dict("index"),
                "duplicates_of_train": duplicates,
                "coco_found_fruit_share": float(test.fruit_box_share.notna().mean()),
                "median_fruit_size_share": float(test.fruit_box_share.median())},
    "classes": CLASSES, "rotten_classes": ROTTEN, "majority_class": majority, "pred_labels": PRED_LABELS,
    "training": {"model": "yolo11n-cls.pt", "epochs": EPOCHS, "imgsz": IMGSZ, "batch": BATCH, "seed": SEED},
    "overall": overall.to_dict("records"), "confusion": confusion,
    "assumptions": {"cost": COST, "base_rate": BASE_RATE, "fruit_per_day": FRUIT_PER_DAY,
                    "seconds_per_review": SECONDS_PER_REVIEW, "wage_per_hour": WAGE_PER_HOUR,
                    "shift_hours": SHIFT_HOURS, "reviewers_available": REVIEWERS_AVAILABLE,
                    "review_cost": REVIEW_COST, "max_review_share": MAX_REVIEW_SHARE},
    "threshold": THRESHOLD, "band": {"low": LOW, "high": HIGH},
    "policies": {name: table.to_dict("records") for name, table in policy_tables.items()},
}
(OUT / "metrics.json").write_text(json.dumps(summary, indent=2, default=float))
print("saved:", sorted(f.name for f in OUT.iterdir()), "and", WEIGHTS)